# Generation Data Integrity Check
Checks for block_id (ud_idx) collisions and data consistency for the Generation Task.

In [1]:
import sys
import os
import pickle
import numpy as np
from collections import defaultdict, Counter

# Add root
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
# Load Data
data_path = "../data/gen_task/processed_gen_data.pkl"
print(f"Loading {data_path}...")

with open(data_path, "rb") as f:
    data, max_len = pickle.load(f)

print(f"Total Samples: {len(data['emotion'])}")

Loading ../data/gen_task/processed_gen_data.pkl...
Total Samples: 51239


In [3]:
# Check 1: Unique Key (ud_idx, ld_idx) Collision
# Every sample should have a unique combination of Dialogue ID and Turn ID.

seen_keys = set()
collisions = []
ud_idx_counter = Counter()

for i in range(len(data['ud_idx'])):
    u = data['ud_idx'][i]
    l = data['ld_idx'][i]
    
    key = (u, l)
    if key in seen_keys:
        collisions.append((i, key))
    else:
        seen_keys.add(key)
        
    ud_idx_counter[u] += 1

print(f"Total Unique (ud_idx, ld_idx) pairs: {len(seen_keys)}")
print(f"Collisions detected: {len(collisions)}")

if len(collisions) > 0:
    print("Example Collisions:", collisions[:10])
else:
    print("✅ No (ud_idx, ld_idx) collisions found.")

Total Unique (ud_idx, ld_idx) pairs: 51239
Collisions detected: 0
✅ No (ud_idx, ld_idx) collisions found.


---
## Split Leakage Check: FSL vs SSL Scenarios

Checks whether any `ud_idx` (dialogue block) appears in more than one split.

**Call flow (no model needed):**
1. Load tokenizer only via `GenModelLoader.load_tokenizer_only()`
2. Build `SFTTrainerConfig` with the desired split settings
3. Call `gen_loader_warp` → returns `(train_loader, val_loader, test_loader, raw_ds)`
4. Extract `ud_idx` sets from each `GenerationDataset.block_map` — **no tokenization triggered**
5. Report intersections

**Two scenarios:**
| Scenario | Train | Val | Test |
|---|---|---|---|
| FSL | 32 dialogues per emotion | 20% of total | 20% of total |
| SSL | 20% of total (semi) | 40% of total | 40% of total |

In [4]:
from ZGeneration.model_loader_gen import GenModelLoader
from ZGeneration.data_loader_gen import gen_loader_warp
from ZGeneration.config_gen import GenTrainingConfig

# Load tokenizer only — no model weights loaded, fast and memory-free
MODEL_NAME = "Qwen3-4B-Instruct-2507"
loader = GenModelLoader(model_path=MODEL_NAME, device="cpu")
tokenizer = loader.load_tokenizer_only()

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Vocab size: {tokenizer.vocab_size}")

/home/new/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model found locally at: /home/new/Documents/LLModel/Qwen3-4B-Instruct-2507
Tokenizer loaded: Qwen2TokenizerFast
Vocab size: 151643


In [5]:
def get_ud_idx_set(gen_dataset) -> set:
    """
    Extract the set of unique ud_idx (dialogue block ids) from a GenerationDataset.
    Uses block_map directly — no __getitem__ call, zero tokenization cost.
    """
    return set(gen_dataset.block_map.keys())


def check_leakage(name: str, train_ds, val_ds, test_ds):
    """
    Print a full leakage report for one scenario.
    Leakage = any ud_idx appearing in more than one split.
    """
    train_ids = get_ud_idx_set(train_ds)
    val_ids   = get_ud_idx_set(val_ds)
    test_ids  = get_ud_idx_set(test_ds)

    train_val  = train_ids & val_ids
    train_test = train_ids & test_ids
    val_test   = val_ids   & test_ids
    all_three  = train_ids & val_ids & test_ids

    print(f"\n{'='*55}")
    print(f"  Scenario: {name}")
    print(f"{'='*55}")
    print(f"  Blocks  — Train: {len(train_ids):4d} | Val: {len(val_ids):4d} | Test: {len(test_ids):4d}")
    print(f"  Samples — Train: {len(train_ds):4d} | Val: {len(val_ds):4d} | Test: {len(test_ds):4d}")
    print(f"  {'─'*51}")
    
    def report(label, overlap):
        status = "✅ No leakage" if len(overlap) == 0 else f"❌ LEAKAGE ({len(overlap)} blocks)"
        print(f"  {label:<22}: {status}")
        if overlap:
            shown = sorted(list(overlap))[:10]
            print(f"    Example ud_idx: {shown}")

    report("Train ∩ Val",        train_val)
    report("Train ∩ Test",       train_test)
    report("Val   ∩ Test",       val_test)
    report("Train ∩ Val ∩ Test", all_three)
    print(f"{'='*55}")
    
    return {
        "train_val":   train_val,
        "train_test":  train_test,
        "val_test":    val_test,
    }


print("Helper functions ready.")

Helper functions ready.


### Scenario A — Few-Shot Learning (FSL)
- **Train**: 32 dialogues per emotion class (sampled by `sample_few_shot_blocks`)
- **Val**: 20% of total blocks (`val_ratio=0.2`)
- **Test**: 20% of total blocks (`test_ratio=0.2`)

In [6]:
# ── Scenario A: FSL ──────────────────────────────────────────────────────────
cfg_fsl = GenTrainingConfig()
cfg_fsl.data_path       = "../data"
cfg_fsl.prompt_key      = "input_text"
cfg_fsl.max_seq_length  = max_len      # reuse max_len loaded in cell 2
cfg_fsl.batch_size      = 1
cfg_fsl.num_workers     = 0

cfg_fsl.few_shot        = True
cfg_fsl.semi_supervised = False
cfg_fsl.shots_per_class = 32           # 32 dialogues per emotion class

cfg_fsl.val_ratio       = 0.2          # 20% of total blocks for val
cfg_fsl.test_ratio      = 0.2          # 20% of total blocks for test

cfg_fsl.fast_train      = False

_, _, _, raw_ds_fsl = gen_loader_warp(data, tokenizer, cfg_fsl)
train_fsl, val_fsl, test_fsl = raw_ds_fsl

leakage_fsl = check_leakage("FSL (32-shot per class)", train_fsl, val_fsl, test_fsl)

Few-shot采样 (block级别): 1024 blocks (32 blocks/class)
Gen Data Split: Train 1024, Val 4966, Test 4966 blocks.

  Scenario: FSL (32-shot per class)
  Blocks  — Train: 1024 | Val: 4966 | Test: 4966
  Samples — Train: 2114 | Val: 10273 | Test: 10109
  ───────────────────────────────────────────────────
  Train ∩ Val           : ✅ No leakage
  Train ∩ Test          : ✅ No leakage
  Val   ∩ Test          : ✅ No leakage
  Train ∩ Val ∩ Test    : ✅ No leakage


In [11]:
len(get_ud_idx_set(train_fsl)) == 32*32

True

### Scenario B — Semi-Supervised Learning (SSL)
- **Train : Val : Test = 1 : 2 : 2** (expressed as fraction of total blocks)
- `semi_ratio=0.2` → train = 20% of total
- `val_ratio=0.4`  → val  = 40% of total
- `test_ratio=0.4` → test = 40% of total

In [7]:
# ── Scenario B: SSL (train:val:test = 1:2:2) ─────────────────────────────────
cfg_ssl = GenTrainingConfig()
cfg_ssl.data_path       = "../data"
cfg_ssl.prompt_key      = "input_text"
cfg_ssl.max_seq_length  = max_len
cfg_ssl.batch_size      = 1
cfg_ssl.num_workers     = 0

cfg_ssl.few_shot        = False
cfg_ssl.semi_supervised = True
cfg_ssl.semi_ratio      = 0.2          # train = 20% of all blocks  (1 part)

cfg_ssl.val_ratio       = 0.4          # val   = 40% of all blocks  (2 parts)
cfg_ssl.test_ratio      = 0.4          # test  = 40% of all blocks  (2 parts)

cfg_ssl.fast_train      = False

_, _, _, raw_ds_ssl = gen_loader_warp(data, tokenizer, cfg_ssl)
train_ssl, val_ssl, test_ssl = raw_ds_ssl

leakage_ssl = check_leakage("SSL (semi_ratio=0.2, val/test=0.4)", train_ssl, val_ssl, test_ssl)

Semi-supervised采样: 4966 labeled blocks (20.0%)
Gen Data Split: Train 4966, Val 9932, Test 9932 blocks.

  Scenario: SSL (semi_ratio=0.2, val/test=0.4)
  Blocks  — Train: 4966 | Val: 9932 | Test: 9932
  Samples — Train: 10254 | Val: 20425 | Test: 20560
  ───────────────────────────────────────────────────
  Train ∩ Val           : ✅ No leakage
  Train ∩ Test          : ✅ No leakage
  Val   ∩ Test          : ✅ No leakage
  Train ∩ Val ∩ Test    : ✅ No leakage


### Cross-Scenario Comparison
Check whether FSL and SSL share the same val/test blocks — important if you want the two scenarios to be evaluated on an identical held-out set.

In [12]:
# ── Cross-scenario overlap check ─────────────────────────────────────────────
fsl_val_ids  = get_ud_idx_set(val_fsl)
fsl_test_ids = get_ud_idx_set(test_fsl)
ssl_val_ids  = get_ud_idx_set(val_ssl)
ssl_test_ids = get_ud_idx_set(test_ssl)

print("\n" + "="*55)
print("  Cross-Scenario Overlap")
print("="*55)

pairs = [
    ("FSL-Val   ∩ SSL-Val",  fsl_val_ids,  ssl_val_ids),
    ("FSL-Test  ∩ SSL-Test", fsl_test_ids, ssl_test_ids),
    ("FSL-Val   ∩ SSL-Test", fsl_val_ids,  ssl_test_ids),
    ("FSL-Test  ∩ SSL-Val",  fsl_test_ids, ssl_val_ids),
]
for label, a, b in pairs:
    overlap = a & b
    pct_a = len(overlap) / len(a) * 100 if a else 0
    pct_b = len(overlap) / len(b) * 100 if b else 0
    msg = "✅ identical" if overlap == a == b else f"{len(overlap)} shared blocks ({pct_a:.1f}% of FSL, {pct_b:.1f}% of SSL)"
    print(f"  {label:<28}: {msg}")

print("="*55)
print()
print("Note: FSL val/test are 20%/20% of total; SSL val/test are 40%/40%.")
print("Expect SSL val/test to be SUPERSETS of FSL val/test (not identical).")


  Cross-Scenario Overlap
  FSL-Val   ∩ SSL-Val         : 3958 shared blocks (79.7% of FSL, 39.9% of SSL)
  FSL-Test  ∩ SSL-Test        : 0 shared blocks (0.0% of FSL, 0.0% of SSL)
  FSL-Val   ∩ SSL-Test        : 0 shared blocks (0.0% of FSL, 0.0% of SSL)
  FSL-Test  ∩ SSL-Val         : 3956 shared blocks (79.7% of FSL, 39.8% of SSL)

Note: FSL val/test are 20%/20% of total; SSL val/test are 40%/40%.
Expect SSL val/test to be SUPERSETS of FSL val/test (not identical).
